In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import product
import scipy.stats as sc
import seaborn as sns
from math import floor

In [ ]:
class ECM:
  '''
  The implementation of ECM (Error Correction Model) estimation for panel data.
  ----
  ----
  PARAMETERS:
  ----
  - *df*: a Pandas DataFrame containing panel data. Make sure your data is structured in this exact order (by column index):\n
    0 - Spatial units. The data must be homogenous, e.g. only cities, contries, regions, etc.\n
    1 - Temporal units. The data must be homogenous, e.g. only years, months, quarters, etc. \n
    2 - Your target/endogenous variable. The data must not contain NaN values.\n
    3+ - Your exogenous variables. The data must not contain NaN values.\n
  - *effects*: Specify what effects your long-run model must have. The class will estimate both the short-run & long-run models.
    Enter one of the following keywords: "fix" | "rand". "rand" by default.
  - *trend*: Specify a trend of which order your target variable is. 0 by default - the data contains no trend.
  - *n_lags*: Specify the maximum amount of lags to test on. The estimator will choose the best lang amount based on AIC.
  - *method*: Specify the method of ECM estimation. \n
    Currently implemented methods:\n
    - MG (Mean Group) - Choose this method of you believe all your coefficients may be heterogenous.
    - CCEMG (Common Correlated Effects Mean Group) - Choose this one if you also believe that there is valid cross-sectional dependence in the data. \n
    Choose between the following keywords: ["MG", "CCEMG"]
  ----
  RETURNS:
  --
  A python dictionary (dict) containing:
    - Long-run estimation results | key = "lr_res"
    - ECM (short-run) estimation results | key = "sr_res"
    - A string representing the estimated long-run equation | key = "lr_eq"
    - A string representing the estimated ECM (short-run) equation | key = "sr_eq"
  '''
  def __init__(self, df: pd.DataFrame, effects: str = 'rand', trend: int = 0, n_lags: int = 1, method: str = 'MG') -> None:
    self.__df = df
    self.__eff = effects.lower()
    self.__t = trend
    self.__lag = n_lags
    self.__method = method.lower()
    self.__exog = len(df.columns[3:])
    self.__l =[]
    for i in range(1, self.__exog+1):
      self.__l.append(f'x{i}')
    self.__df.columns = ['SpUnit', 'time', 'target'] + self.__l
    self.__N = len(self.__df.SpUnit.unique())
    self.__T = len(self.__df.time.unique())
    if self.__lag > self.__T/5:
      self.__lag = floor(self.__T/5)
      if self.__lag < 1:
        self.__lag = 1
    self.__lr = self.build_lr()
    self.__sr = None
    self.__verify()
  
  def __verify(self) -> None:
    if self.__eff not in ['fix', 'rand']:
      raise ValueError('Non-Valid panel effects type entered!')
    if self.__trend < 0:
      raise ValueError('The Trend order cannot be lower than 0!')
    if self.__method not in ['mg', 'ccemg']:
      raise NotImplementedError('Either the estimation method has not been implemented yet, or it is invalid!')
  
  def build_lr(self) -> pd.DataFrame:
    lr = self.__df.copy(deep=True)
    if self.__method == 'fix':
      for i, unit in enumerate(lr.SpUnit.unique(), start=1):
        lr[f'd{i}'] = np.where(lr.SpUnit == unit, 1, 0)
      return lr
    else:
      pass
  def __del__(self) -> None:
    pass